In [7]:
import sys
import os
import warnings
from datetime import datetime, timezone, timedelta
from pathlib import Path
import pandas as pd
import numpy as np
import scipy.stats as stats
import matplotlib.pyplot as plt
import seaborn as sns
import pytz
import json
from zoneinfo import ZoneInfo
from pandas import Timestamp
from statsmodels.tsa.stattools import adfuller, kpss, acf, pacf
from statsmodels.graphics.tsaplots import plot_acf, plot_pacf
from statsmodels.stats.diagnostic import acorr_ljungbox
from statsmodels.tsa.arima.model import ARIMA
from sklearn.model_selection import TimeSeriesSplit
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.model_selection import TimeSeriesSplit
from sklearn.metrics import r2_score
from scipy.stats import norm

current_dir = Path.cwd()
parent_dir = current_dir.parent
sys.path.insert(0, str(parent_dir))
from lib import *

MODEL_PATH=parent_dir / 'models' 
warnings.filterwarnings('ignore')
pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', 1000)

In [8]:
example_ticker = "KXBTC15M-26MAY040515-15"
lookback_minutes = 10000
series, event_dt = parse_kalshi_15m_event_ticker(example_ticker)
dt_only = get_ticker_datetime(example_ticker)
# crypto_at = datetime(2026,5,5,0,0,tzinfo=ZoneInfo('America/Chicago'))
crypto_at = datetime.now(tz=ZoneInfo('America/Chicago'))
df_api = get_market_data_from_api(series, crypto_at, lookback_minutes)
df_api = df_api.set_index('datetime')
df_api.head()

API Error Response: {'error': {'code': 'not_found', 'message': 'not found', 'service': 'query-exchange'}}
Error getting market candlesticks: 404 Client Error: Not Found for url: https://api.elections.kalshi.com/trade-api/v2/series/KXBTC15M/markets/KXBTC15M-26MAY280215-15/candlesticks?series_ticker=KXBTC15M&ticker=KXBTC15M-26MAY280215-15&start_ts=1779947700&end_ts=1779948900&period_interval=1&include_latest_before_start=True&limit=1000
API Error Response: {'error': {'code': 'not_found', 'message': 'not found', 'service': 'query-exchange'}}
Error getting market candlesticks: 404 Client Error: Not Found for url: https://api.elections.kalshi.com/trade-api/v2/series/KXBTC15M/markets/KXBTC15M-26MAY280230-30/candlesticks?series_ticker=KXBTC15M&ticker=KXBTC15M-26MAY280230-30&start_ts=1779948600&end_ts=1779949800&period_interval=1&include_latest_before_start=True&limit=1000
API Error Response: {'error': {'code': 'not_found', 'message': 'not found', 'service': 'query-exchange'}}
Error getting ma

,ticker,floor_strike,volume_fp,open_interest_fp,yes_ask_open_dollar,yes_ask_high_dollar,yes_ask_low_dollar,yes_ask_close_dollar,yes_bid_open_dollar,yes_bid_high_dollar,yes_bid_low_dollar,yes_bid_close_dollar,no_ask_open_dollar,no_ask_high_dollar,no_ask_low_dollar,no_ask_close_dollar,no_bid_open_dollar,no_bid_high_dollar,no_bid_low_dollar,no_bid_close_dollar
datetime,,,,,,,,,,,,,,,,,,,,
2026-05-22 09:16:00-05:00,KXBTC15M-26MAY221030-30,76971.75,26668.23,16314.64,0.999,0.999,0.52,0.52,0.00,0.66,0.00,0.51,0.49,0.34,1.00,1.00,0.48,0.001,0.48,0.001
2026-05-22 09:17:00-05:00,KXBTC15M-26MAY221030-30,76971.75,34165.98,46070.57,0.520,0.520,0.46,0.49,0.51,0.51,0.45,0.48,0.52,0.49,0.55,0.49,0.51,0.480,0.54,0.480
2026-05-22 09:18:00-05:00,KXBTC15M-26MAY221030-30,76971.75,25448.45,60943.77,0.490,0.530,0.45,0.46,0.48,0.51,0.44,0.45,0.55,0.49,0.56,0.52,0.54,0.470,0.55,0.510
2026-05-22 09:19:00-05:00,KXBTC15M-26MAY221030-30,76971.75,46646.89,82445.21,0.460,0.460,0.24,0.28,0.45,0.45,0.23,0.27,0.73,0.55,0.77,0.55,0.72,0.540,0.76,0.540
2026-05-22 09:20:00-05:00,KXBTC15M-26MAY221030-30,76971.75,37881.17,105416.56,0.280,0.310,0.20,0.24,0.27,0.29,0.19,0.23,0.77,0.71,0.81,0.73,0.76,0.690,0.80,0.720


In [11]:
df_crypto = get_crypto_past_minutes(series, crypto_at, lookback_minutes)
df_crypto['datetime'] = pd.to_datetime(df_crypto['datetime'])
df_crypto['datetime'] = df_crypto['datetime'].dt.tz_convert('America/Chicago')
df_crypto['datetime'] = df_crypto['datetime'].dt.floor('min')
df_crypto = df_crypto.set_index('datetime')
filter_timestamp = df_crypto[df_crypto.index.minute.isin([0,15,30,45])].index[0]
df_crypto = df_crypto[df_crypto.index >= filter_timestamp]
df_crypto.head()

ResourceExhausted: 429 Quota exceeded.

In [ ]:
df_merged = df_crypto.join(df_api, how='left')
df_merged = df_merged.dropna()
df_merged.head()

In [ ]:
df_calc = df_merged

In [ ]:
def rsi_calc(series, periods=14):
    delta = series.diff()
    gain = (delta.where(delta>0,0)).rolling(periods).mean()
    loss = (delta.where(delta<0,0)).rolling(periods).mean()
    rs = gain / loss
    rsi = 100 - (100/(1+rs))
    return rsi

In [ ]:
def macd_calc(series, fast=14, slow=26, signal=13):
    ema_fast = series.ewm(span=fast, adjust=False).mean()
    ema_slow = series.ewm(span=slow, adjust=False).mean()
    macd_line = ema_fast - ema_slow
    signal_line = macd_line.ewm(span=signal, adjust=False).mean()
    histogram = macd_line - signal_line
    return macd_line, signal_line, histogram

In [ ]:
research_side = 'no'
df_calc['rsi'] = rsi_calc(df_calc['close'], periods=5) 
df_calc['macd'], df_calc['signal_line'], _ = macd_calc(df_calc['close'], 4, 9, 2) 
df_calc[research_side + '_dist'] = df_calc['close'] - df_calc['floor_strike']
df_calc['log_return'] = np.log(df_calc['close'] / df_calc['close'].shift(1))
df_calc['m3_log_return'] = df_calc['log_return'].rolling(3).std()
df_calc['m5_log_return'] = df_calc['log_return'].rolling(5).std()
df_calc['ma3'] = df_calc['close'].rolling(3).mean()
df_calc['ma5'] = df_calc['close'].rolling(5).mean()
df_calc['ma3_vs_strike'] = (df_calc['ma3'] - df_calc['floor_strike'])/df_calc['floor_strike'] * 100
df_calc['ma5_vs_strike'] = (df_calc['ma5'] - df_calc['floor_strike'])/df_calc['floor_strike'] * 100
df_calc[research_side + '_dist_pct'] = df_calc[research_side + '_dist'] / df_calc['floor_strike'] * 100
df_calc['m1_' + research_side + '_dist_momentum'] = df_calc[research_side + '_dist'] - df_calc[research_side + '_dist'].shift(1)
df_calc['m3_' + research_side + '_dist_momentum'] = df_calc[research_side + '_dist'] - df_calc[research_side + '_dist'].shift(3)
df_calc['m5_' + research_side + '_dist_momentum'] = df_calc[research_side + '_dist'] - df_calc[research_side + '_dist'].shift(5)
df_calc['time_decay'] = np.where(df_calc.index.minute % 15 == 0, 0, 15 - df_calc.index.minute % 15)
df_calc['hour'] = df_calc.index.hour
df_calc['minute'] = df_calc.index.minute
df_calc[research_side + '_spread'] = df_calc[research_side + '_ask_close_dollar'] - df_calc[research_side + '_bid_close_dollar']
df_calc['volume_surge'] = df_calc['volume_fp'] / df_calc['volume_fp'].rolling(5).mean()
df_calc['ask_delta'] = df_calc[research_side + '_ask_low_dollar'].diff()
df_calc['bid_delta'] = df_calc[research_side + '_bid_high_dollar'].diff()
df_calc['ask_std'] = df_calc[research_side + '_ask_low_dollar'].rolling(3).std()
df_calc['bid_std'] = df_calc[research_side + '_bid_high_dollar'].rolling(3).std()
# df_calc['oi_change'] = df_calc['open_interest_fp'] - df_calc['open_interest_fp'].shift(1)

In [ ]:
df_calc = df_calc.dropna()

In [ ]:
df_calc.head()

In [ ]:
def agg_data_function(df, column, *data_cents):
    results = {cent: [] for cent in data_cents}
    triggered = set() 
    
    for i in range(len(df)):
        row = df.iloc[i]
        
        if row['time_decay'] == 0:
            continue
        
        current_ticker = row['ticker']
        
        for cent in data_cents:
            if (current_ticker, cent) in triggered:
                continue
                
            if float(row[column + '_ask_low_dollar']) < float(cent):
                # print(f"trigger: {column + '_ask_low_dollar'}: {row[column + '_ask_low_dollar']}")
                triggered.add((current_ticker, cent)) 
                
                high_price = 0
                low_price = 1
                
                for j in range(1, 16):
                    if i + j >= len(df):
                        break
                    next_row = df.iloc[i + j]
                    high_price = max(next_row[column + '_bid_high_dollar'], high_price)
                    low_price = min(next_row[column + '_ask_low_dollar'], low_price)
                    if next_row['ticker'] != current_ticker or next_row['time_decay'] == 0:
                        break
                
                tmp_dict = row.to_dict()
                tmp_dict['subsequent_high'] = high_price
                tmp_dict['subsequent_low'] = low_price
                tmp_dict['reached_30'] = 1 if high_price >= 0.30 else 0
                tmp_dict['reached_40'] = 1 if high_price >= 0.40 else 0
                tmp_dict['reached_50'] = 1 if high_price >= 0.50 else 0
                tmp_dict['reached_60'] = 1 if high_price >= 0.60 else 0
                tmp_dict['reached_70'] = 1 if high_price >= 0.7 else 0
                tmp_dict['reached_90'] = 1 if high_price >= 0.9 else 0
                tmp_dict['reached_99'] = 1 if high_price >= 0.99 else 0
                results[cent].append(tmp_dict)
                
    
    return results

In [ ]:
res=agg_data_function(df_calc, research_side, *[0.05,0.1,0.15,0.2,0.25,0.3,0.35])

In [ ]:
# win rate analysis
comb_05 = pd.DataFrame(res[0.05])
comb_10 = pd.DataFrame(res[0.10])
comb_15 = pd.DataFrame(res[0.15])
comb_20 = pd.DataFrame(res[0.2])
comb_25 = pd.DataFrame(res[0.25])
comb_30 = pd.DataFrame(res[0.3])
comb_35 = pd.DataFrame(res[0.35])

In [ ]:
def get_protential_pnl(df, entry_price, *exit_price):
    for price in exit_price:
        col_name = f"reached_{int(price * 100)}"
        if col_name in df.columns:
            rate = (df[col_name] == 1).sum() / len(df)
            pnl = rate * (float(price) - float(entry_price))
            print(f"Entry: {entry_price}, Exit: {price}, win rate: {rate:.2%}, PNL: {pnl:.4f}")

In [ ]:
def evaluate_feature(good_mean, good_std, bad_mean, bad_std):
    diff = good_mean - bad_mean
    avg_std = (good_std + bad_std) / 2
    
    if avg_std < 1e-10:
        return None
    
    ratio = abs(diff) / avg_std
    
    if ratio > 0.5:
        grade = 'strong'
    elif ratio > 0.3:
        grade = 'medium'
    elif ratio > 0.15:
        grade = 'weak'
    else:
        grade = 'useless'
    
    return ratio, grade

In [ ]:
good_hours = [9,10,11,12]

get_protential_pnl(comb_05[comb_05['hour'].isin(good_hours)],0.05,*[0.4,0.5,0.6,0.7,0.9,0.99])
get_protential_pnl(comb_10[comb_10['hour'].isin(good_hours)],0.10,*[0.4,0.5,0.6,0.7,0.9,0.99])
get_protential_pnl(comb_15[comb_15['hour'].isin(good_hours)],0.15,*[0.4,0.5,0.6,0.7,0.9,0.99])
get_protential_pnl(comb_20[comb_20['hour'].isin(good_hours)],0.20,*[0.4,0.5,0.6,0.7,0.9,0.99])
get_protential_pnl(comb_25[comb_25['hour'].isin(good_hours)],0.25,*[0.4,0.5,0.6,0.7,0.9,0.99])
get_protential_pnl(comb_30[comb_30['hour'].isin(good_hours)],0.30,*[0.4,0.5,0.6,0.7,0.9,0.99])
get_protential_pnl(comb_35[comb_35['hour'].isin(good_hours)],0.35,*[0.4,0.5,0.6,0.7,0.9,0.99])

In [ ]:
feature_cols = [  
    'ask_delta',
    'bid_delta',
    'ask_std',
    'bid_std',
    research_side + '_dist',         
    'log_return',  
    'm3_log_return',
    'm5_log_return',  
    'ma3_vs_strike',
    'ma5_vs_strike',            
    research_side + '_dist_pct',
    'm1_' + research_side + '_dist_momentum',
    'm3_' + research_side + '_dist_momentum',
    'm5_' + research_side + '_dist_momentum',
    'time_decay',
    'hour',
    'minute',
    research_side + '_spread',
    'volume_surge',
    'rsi',
    'macd',
    'signal_line',
]

comb_list = [0.05, 0.1, 0.15, 0.2, 0.25,0.3,0.35]
reached_list = ['reached_30','reached_40', 'reached_50', 'reached_60', 'reached_70', 'reached_90', 'reached_99']

for comb in comb_list: 
    for reached in reached_list:
        df_results = pd.DataFrame(res[comb])
        df_results = df_results[df_results['hour'].isin(good_hours)]
        df_expected = df_results[reached]
        
        # Separate good and bad outcomes
        good = df_results[df_expected == 1]
        bad = df_results[df_expected == 0]
        
        # Prior probabilities
        prior_good = len(good) / len(df_results)
        prior_bad = len(bad) / len(df_results)
        print(f"Comb is {comb}, reached is {reached}")
        print(f"Prior P(Good) = {prior_good:.4f}, P(Bad) = {prior_bad:.4f}")
        likelihoods = {}
        good_ratio_count = 0
        for col in feature_cols:
            good_mean, good_std = good[col].mean(), good[col].std()
            bad_mean, bad_std = bad[col].mean(), bad[col].std()
            
            likelihoods[col] = {
                'good': (good_mean, good_std),
                'bad': (bad_mean, bad_std),
            }
            
            diff = good_mean - bad_mean
            ratio, grade = evaluate_feature(good_mean, good_std, bad_mean, bad_std)
            if grade == 'strong' or grade == 'medium':
                good_ratio_count += 1
            print(f"{grade} -> {col}: Good={good_mean:.4f}, Bad={bad_mean:.4f}, ratio={ratio:.4f}")
        print(f"Total Good Ratio is {good_ratio_count}\n")

In [10]:
df_name = 0.10
df_result_name = 'reached_90'

df_results = pd.DataFrame(res[df_name])
df_results = df_results[df_results['hour'].isin(good_hours)]
df_expected = df_results[df_result_name]

# Separate good and bad outcomes
good = df_results[df_expected == 1]
bad = df_results[df_expected == 0]

# Prior probabilities
prior_good = len(good) / len(df_results)
prior_bad = len(bad) / len(df_results)
likelihoods = {}
params = {}

for col in feature_cols:
    good_mean, good_std = good[col].mean(), good[col].std()
    bad_mean, bad_std = bad[col].mean(), bad[col].std()
    
    likelihoods[col] = {
        'good': (good_mean, good_std),
        'bad': (bad_mean, bad_std),
    }
    
    # diff = good_mean - bad_mean
    ratio, grade = evaluate_feature(good_mean, good_std, bad_mean, bad_std)
    if grade == 'strong' or grade == 'medium':
        print(f"{grade} -> {col}: Good={good_mean:.4f}, Bad={bad_mean:.4f}, Ratio={ratio:.4f}")
        params[col] = {'good': (good_mean, good_std), 'bad': (bad_mean, bad_std),}
# for col in feature_cols:
#     g_mean, g_std = likelihoods[col]['good']
#     b_mean, b_std = likelihoods[col]['bad']
#     params[col] = {
#         'good': (g_mean, g_std),
#         'bad': (b_mean, b_std)
#     }
    
params['period'] = {
    'good': (prior_good,),
    'bad': (prior_bad,)
}
print(f"good is: {prior_good}, bad is: {prior_bad}")

NameError: name 'res' is not defined

In [61]:
# type 1 and 2 errors

type1 = 0 
type2 = 0  

total_good = 0
total_bad = 0

feature_names = [name for name in params.keys() if name != 'period']
def predict(*args):
    # Start from prior odds
    log_odds = np.log(params['period']['good'][0] / params['period']['bad'][0])
    
    for name, x in zip(feature_names, args):
        g_m, g_s = params[name]['good']
        b_m, b_s = params[name]['bad']
        
        # Likelihood ratio
        p_g = norm.pdf(x, g_m, g_s + 1e-10)
        p_b = norm.pdf(x, b_m, b_s + 1e-10)
        
        if p_b > 0:
            log_odds += np.log(p_g / p_b)
    
    prob = 1 / (1 + np.exp(-log_odds))
    return prob

for threshold in [0.1, 0.15, 0.2, 0.25, 0.30, 0.35, 0.40, 0.45]:
    type1, type2 = 0, 0
    total_good, total_bad = 0, 0
    
    for index, row in df_results.iterrows(): 
        p = predict(*(row[x] for x in params if x != 'period')) 
        
        actual = row[df_result_name]
        
        if actual == 1:
            total_good += 1
            if p < threshold:
                type1 += 1
        if actual == 0:
            total_bad += 1
            if p >= threshold:
                type2 += 1
    
    bought_good = total_good - type1
    bought_bad = type2
    
    type1_rate = type1 / total_good
    type2_rate = type2 / total_bad
    
    # Precision & Recall
    precision = bought_good / (bought_good + bought_bad) if (bought_good + bought_bad) > 0 else 0
    recall = 1 - type1_rate
    
    # F1
    f1 = 2 * precision * recall / (precision + recall) if (precision + recall) > 0 else 0
    
    print(f"Threshold={threshold:.2f}: Type1={type1_rate:.2f}, Type2={type2_rate:.2f}, F1={f1:.4f}, Precision={precision:.4f}, Recall={recall:.4f}")

Threshold=0.10: Type1=0.00, Type2=0.62, F1=0.4615, Precision=0.3000, Recall=1.0000
Threshold=0.15: Type1=0.00, Type2=0.53, F1=0.5000, Precision=0.3333, Recall=1.0000
Threshold=0.20: Type1=0.11, Type2=0.41, F1=0.5161, Precision=0.3636, Recall=0.8889
Threshold=0.25: Type1=0.11, Type2=0.38, F1=0.5333, Precision=0.3810, Recall=0.8889
Threshold=0.30: Type1=0.11, Type2=0.32, F1=0.5714, Precision=0.4211, Recall=0.8889
Threshold=0.35: Type1=0.22, Type2=0.29, F1=0.5385, Precision=0.4118, Recall=0.7778
Threshold=0.40: Type1=0.22, Type2=0.29, F1=0.5385, Precision=0.4118, Recall=0.7778
Threshold=0.45: Type1=0.33, Type2=0.18, F1=0.5714, Precision=0.5000, Recall=0.6667


In [62]:
# export model

def write_model_to_json(parameters: dict, filepath: str = None):
    if filepath is None:
        filepath = MODEL_PATH / 'yes_bayesian.json'
    
    params_serializable = {}
    for feature, values in parameters.items():
        params_serializable[feature] = {
            'good': list(values['good']),
            'bad': list(values['bad'])
        }
    
    with open(filepath, 'w') as f:
        json.dump(params_serializable, f, indent=2)
    
    print(f"Model saved to {filepath}")


def load_model_from_json(filepath: str = None):
    if filepath is None:
        filepath = MODEL_PATH / 'yes_bayesian.json'
    
    with open(filepath, 'r') as f:
        params_serializable = json.load(f)
    
    # Convert lists back to tuples
    parameters = {}
    for feature, values in params_serializable.items():
        parameters[feature] = {
            'good': tuple(values['good']),
            'bad': tuple(values['bad'])
        }
    
    return parameters


# Correct dict syntax
# params = {
#     'period': {
#         'good': (0.4138,),
#         'bad': (0.5862,)
#     },
#     'm1_yes_dist_momentum': {
#         'good': (-17.5070, 51.8947),
#         'bad':  (-19.7738, 48.4898)
#     },
#     'm3_yes_dist_momentum': {
#         'good': (-29.0380, 82.5982),
#         'bad':  (-32.8337, 73.5574)
#     },
#     'm5_yes_dist_momentum': {
#         'good': (-31.1500, 93.6206),
#         'bad':  (-37.5775, 85.6289)
#     },
#     'yes_dist': {
#         'good': (-19.9004, 29.5222),
#         'bad':  (-22.8592, 38.3313)
#     },
# }

# Write
write_model_to_json(params)
print(params)

Model saved to /Users/yingxie/Documents/Git/Quant/Kalshi/btc_15_strategy/models/yes_bayesian.json
{'bid_delta': {'good': (-0.15222222222222223, 0.14228297313608698), 'bad': (-0.10647058823529412, 0.13425173895247952)}, 'yes_dist': {'good': (-56.73888888888986, 51.38984200317516), 'bad': (-88.55176470588134, 85.42980561666124)}, 'm3_log_return': {'good': (0.0003197268710172076, 0.000196562368782156), 'bad': (0.0004110608988313041, 0.0003246869075403677)}, 'ma3_vs_strike': {'good': (-0.05035352266128644, 0.06889022137715817), 'bad': (-0.0922487663763957, 0.08959982626574327)}, 'ma5_vs_strike': {'good': (-0.03533346053291394, 0.05539471560862602), 'bad': (-0.07286415743777037, 0.07337570496825947)}, 'yes_dist_pct': {'good': (-0.07435120253943993, 0.06714480014997021), 'bad': (-0.11555831698797336, 0.11136012359192875)}, 'time_decay': {'good': (5.777777777777778, 2.6352313834736494), 'bad': (4.882352941176471, 2.847270951570601)}, 'hour': {'good': (10.88888888888889, 1.2692955176439846), '

In [23]:
# hour research
reached_list = [50,60,70,90,99]
time_dict = {str(x): 0 for x in range(24)}
test_df = comb_10
test_entry_price = 10

for hour in range(24):
    for reached in reached_list:
        df_h = test_df[test_df['hour'] == hour]
        # if len(df_h) < 2:
        #     continue
        rate = df_h['reached_' + str(reached)].mean()
        ev = rate * ((reached - test_entry_price) / 100)  - (1-rate) * test_entry_price / 100
        time_dict[str(hour)] += ev

sorted_items = sorted(time_dict.items(), key=lambda x: int(x[0]), reverse=True)
for key, item in sorted_items:
    print(f"Hour: {key}, EV: {item:.3f}")

Hour: 23, EV: nan
Hour: 22, EV: nan
Hour: 21, EV: nan
Hour: 20, EV: 1.345
Hour: 19, EV: -0.500
Hour: 18, EV: -0.500
Hour: 17, EV: -0.336
Hour: 16, EV: -0.001
Hour: 15, EV: -0.043
Hour: 14, EV: -0.500
Hour: 13, EV: -0.500
Hour: 12, EV: 1.140
Hour: 11, EV: 0.288
Hour: 10, EV: -0.131
Hour: 9, EV: 0.085
Hour: 8, EV: -0.193
Hour: 7, EV: 0.670
Hour: 6, EV: -0.165
Hour: 5, EV: -0.036
Hour: 4, EV: -0.122
Hour: 3, EV: -0.236
Hour: 2, EV: -0.148
Hour: 1, EV: nan
Hour: 0, EV: nan


In [24]:
# start research
trade_hour = [x for x in range(23)]
reached_dict = {str(x): 0 for x in reached_list}
for hour in trade_hour:
    for reached in reached_list:
        df_h = test_df[test_df['hour'] == hour]
        if len(df_h) < 5:
            continue
        rate = df_h['reached_' + str(reached)].mean()
        ev = rate * ((reached - test_entry_price)/100 - 0.1)  - (1-rate) * test_entry_price / 100
        reached_dict[str(reached)] += ev

for key, item in reached_dict.items():
    print(f"Reached: {key}, EV: {item:.3f}")

Reached: 50, EV: -0.383
Reached: 60, EV: -0.319
Reached: 70, EV: -0.126
Reached: 90, EV: 0.010
Reached: 99, EV: -0.011


In [36]:
# final hour research
trade_hour = [x for x in range(23)]
for hour in trade_hour:
    df_h = test_df[test_df['hour'] == hour]
    # if len(df_h) < 2:
    #     continue
    rate = df_h['reached_90'].mean()
    ev = rate * (0.99 - 0.1)  - (1-rate) * 0.10
    print(f"{hour:02d}:00 {len(df_h):3d} trades win {rate:.0%} | EV {ev:.2f}")

00:00   0 trades win nan% | EV nan
01:00   0 trades win nan% | EV nan
02:00  17 trades win 6% | EV -0.04
03:00  14 trades win 7% | EV -0.03
04:00  14 trades win 7% | EV -0.03
05:00  17 trades win 12% | EV 0.02
06:00  11 trades win 9% | EV -0.01
07:00  11 trades win 27% | EV 0.17
08:00  12 trades win 8% | EV -0.02
09:00  14 trades win 14% | EV 0.04
10:00  10 trades win 10% | EV -0.00
11:00  10 trades win 20% | EV 0.10
12:00   9 trades win 44% | EV 0.34
13:00  11 trades win 0% | EV -0.10
14:00   4 trades win 0% | EV -0.10
15:00   7 trades win 14% | EV 0.04
16:00  11 trades win 9% | EV -0.01
17:00  11 trades win 0% | EV -0.10
18:00   1 trades win 0% | EV -0.10
19:00   3 trades win 0% | EV -0.10
20:00   2 trades win 50% | EV 0.40
21:00   0 trades win nan% | EV nan
22:00   0 trades win nan% | EV nan
